## Build a RAG System

---

## RAG Architecture Overview

RAG (Retrieval-Augmented Generation) combines the power of vector search with LLMs. Instead of relying solely on the model's training data, RAG **retrieves relevant documents** to ground responses in your actual data—reducing hallucinations and enabling up-to-date answers.

---

## Part 1: Document Chunking

In [1]:
!pip install sentence-transformers datasets faiss-cpu transformers torch -q

from datasets import load_dataset
import pandas as pd

# Load ALL Investopedia financial terms (full dataset, only 13.6 MB)
dataset = load_dataset("infCapital/investopedia_terms_en", split="train")
df = dataset.to_pandas()

print(f"\u2705 Loaded {len(df):,} financial term definitions")
print(f"Columns: {list(df.columns)}")
print(f"\nSample terms: {', '.join(df['name'].sample(5, random_state=42).tolist())}")
df[['name', 'text']].head(3)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.5 MB/s eta 0:00:00:00:0100:01


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/485 [00:00<?, ?B/s]

data/train-00000-of-00001-ff1d56c46173cc(…):   0%|          | 0.00/13.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6305 [00:00<?, ? examples/s]

✅ Loaded 6,305 financial term definitions
Columns: ['name', 'text']

Sample terms: Geographical Labor Mobility Definition, Back Office Definition, Exempt-Interest Dividend Definition, Voluntary Accidental Death And Dismemberment Insurance (VAD&D) Definition, Bundle of Rights Definition


,name,text
0,0x Protocol Definition,\nThe 0x protocol is an open protocol that ena...
1,1%/10 Net 30 Definition,\nThe 1%/10 net 30 calculation is a way of pro...
2,10-K Definition,\nA 10-K is a comprehensive report filed annua...


In [2]:
import textwrap

def wrap(text, width=80, indent="   "):
    """Word-wrap text for readable output."""
    return textwrap.fill(text, width=width, initial_indent=indent,
                         subsequent_indent=indent)

def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    text = str(text).strip()
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    return chunks

# Chunk all documents — prepend article name for better retrieval
all_chunks = []
chunk_to_doc = []    # Track which document each chunk came from
doc_first_chunk = {} # Map doc_id -> index of its FIRST chunk (contains definition)

for idx, row in df.iterrows():
    name = row['name']
    chunks = chunk_text(row['text'])
    # Record where this article's first chunk lives
    doc_first_chunk[idx] = len(all_chunks)
    # Prepend article name so embeddings capture the topic
    named_chunks = [f"[{name}] {chunk}" for chunk in chunks]
    all_chunks.extend(named_chunks)
    chunk_to_doc.extend([idx] * len(chunks))

print("\U0001f4c4 DOCUMENT CHUNKING")
print("=" * 60)
print(f"Original documents: {len(df):,}")
print(f"Total chunks created: {len(all_chunks):,}")
print(f"Average chunks per document: {len(all_chunks)/len(df):.1f}")
print(f"Chunk size: 500 chars, Overlap: 50 chars")
print(f"\nExample first chunk (contains definition):")
print(wrap(all_chunks[0][:200], width=80))

📄 DOCUMENT CHUNKING
Original documents: 6,305
Total chunks created: 58,919
Average chunks per document: 9.3
Chunk size: 500 chars, Overlap: 50 chars

Example first chunk (contains definition):
   [0x Protocol Definition] The 0x protocol is an open protocol that enables the
   peer-to-peer exchange of assets on the Ethereum blockchain. The 0x protocol
   was launched in 2017. It was built by 0x Labs,


## Part 2: Embedding Index

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load embedding model (better retrieval than MiniLM, 512-token context)
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')

print(f"Encoding {len(all_chunks):,} chunks (3-5 min on T4 GPU, longer on CPU)...")
chunk_embeddings = embedding_model.encode(all_chunks, show_progress_bar=True)

print(f"\n\u2705 Encoded {len(chunk_embeddings):,} chunks")
print(f"Embedding dimension: {chunk_embeddings.shape[1]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 58,919 chunks (3-5 min on T4 GPU, longer on CPU)...


Batches:   0%|          | 0/1842 [00:00<?, ?it/s]


✅ Encoded 58,919 chunks
Embedding dimension: 384


In [4]:
# Build FAISS index
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings).astype('float32'))

print("📊 VECTOR INDEX BUILT")
print("=" * 60)
print(f"Embeddings shape: {chunk_embeddings.shape}")
print(f"Index type: FAISS (Flat L2)")
print(f"Index size: {index.ntotal} vectors")

📊 VECTOR INDEX BUILT
Embeddings shape: (58919, 384)
Index type: FAISS (Flat L2)
Index size: 58919 vectors


## Part 3: Retrieval

In [5]:
def retrieve(query, k=5):
    """Retrieve top-k relevant chunks for a query."""
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(
        np.array(query_embedding).astype('float32'), k
    )

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'chunk': all_chunks[idx],
            'distance': distances[0][i],
            'doc_id': chunk_to_doc[idx]
        })
    return results

# Test retrieval
query = "What is a leveraged buyout?"

print("\U0001f50e RETRIEVAL TEST")
print("=" * 60)
print(f"Query: \"{query}\"\n")

results = retrieve(query)

print("Retrieved Chunks:")
for i, r in enumerate(results, 1):
    score = 1 / (1 + r['distance'])  # Convert distance to similarity
    print(f"\n{i}. [Score: {score:.2f}] (Term #{r['doc_id']})")
    print(wrap(r['chunk'][:200]))

🔎 RETRIEVAL TEST
Query: "What is a leveraged buyout?"

Retrieved Chunks:

1. [Score: 0.78] (Term #2988)
   [Leveraged Buyback Definition] A leveraged buyback, also known as a leveraged
   share repurchase, is a corporate finance transaction that enables a company
   to repurchase some of its shares using debt. B

2. [Score: 0.78] (Term #630)
   [Buy-In Management Buyout (BIMBO) Definition] in Europe to describe a type of
   LBO that combines new external management with internal management to refresh
   the direction of the company and streamline

3. [Score: 0.77] (Term #2989)
   [Leveraged Buyout (LBO) Definition] A leveraged buyout (LBO) is the
   acquisition of another company using a significant amount of borrowed money
   to meet the cost of acquisition. The assets of the compa

4. [Score: 0.77] (Term #641)
   [Buyout Definition] m the buyers, financiers, and sometimes the seller.
   Leveraged buyouts (LBO) use significant amounts of borrowed money, with the
   assets of the compa

### Why Raw Retrieval Isn't Enough: The Re-Ranking Problem

The retrieval step above uses **vector similarity** to find chunks that are semantically close to your query. This is fast (FAISS searches thousands of vectors in milliseconds), but it has a key weakness:

**Similarity != Best Answer.**

Consider what happens when you ask *"What is dollar-cost averaging?"*

| Chunk | Similarity | Content |
|-------|-----------|---------|
| Chunk from middle of DCA article | 0.85 | *"...investors may modify this rate to 1.20..."* |
| **First paragraph of DCA article** | **0.78** | ***"Dollar-cost averaging is the practice of..."*** |

The chunk with the **highest similarity score** talks about DCA tangentially (mentions it in passing while discussing rates). But the chunk with the **actual definition** ranks lower because it uses broader language.

This is a well-known problem in production RAG systems, and the solution is **re-ranking** — a second pass that reorders retrieved results using smarter logic.

### Two-Stage Retrieval

```
Stage 1: FAST RECALL (FAISS)              Stage 2: RE-RANKING (smarter selection)
─────────────────────────                 ──────────────────────────────────────
Search 10,000+ chunks in milliseconds     Take top results, pick the BEST ones
Optimized for NOT MISSING relevant docs   Optimized for PRECISION and QUALITY
Returns ~5 candidates (wide net)          Returns 1-2 chunks (focused context)
```

**In production**, companies use cross-encoder models (like `ms-marco-MiniLM`) that read the query and each chunk together to produce a smarter relevance score.

**In our notebook**, we use a simpler but effective approach: since Investopedia articles always start with the definition, we use FAISS to find the right *article*, then grab its **first chunk** (which contains the definition). This is domain-aware re-ranking — we're using knowledge about our data's structure to improve retrieval.

### Our Approach: Definition-Aware Retrieval

The `rag_answer` function below implements a simple re-ranking strategy:

1. **FAISS retrieves** the top-5 most similar chunks (fast recall)
2. **We identify** which *article* the best chunk came from
3. **We grab that article's first chunk** — which contains the definition
4. If a second article also matched, we include its definition too

```
Query: "What is a hedge fund?"
    │
    ├─ FAISS finds: Chunk #7 from "Hedge Fund Definition" (score: 0.76)
    │
    └─ We look up: First chunk of "Hedge Fund Definition"
         → "A hedge fund is a limited partnership of private investors..."
              ↑ THIS is what gets sent to the LLM
```

This works because Investopedia articles follow a predictable structure — definitions first, details later. Real-world RAG systems exploit this kind of domain knowledge all the time.

## Part 4: Generation with RAG

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Auto-detect GPU (free Colab T4) or fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# flan-t5-large (780M) — much better quality than flan-t5-base (250M)
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def generate_text(prompt, max_new_tokens=150):
    """Generate text using flan-t5-large."""
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = t5_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        no_repeat_ngram_size=3,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Quick smoke test
print(f"\u2705 Language model loaded ({model_name})")
print(f"   Smoke test: \"{generate_text('What is a stock? Answer in one sentence.')}\"")

Using device: cpu


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

✅ Language model loaded (google/flan-t5-large)
   Smoke test: "A stock is a financial instrument used to buy and sell shares in a company."


In [7]:
def rag_answer(query, k=5):
    """Answer a question using RAG with definition-aware retrieval."""
    # Step 1: Retrieve top-k chunks to find the best-matching article
    chunks = retrieve(query, k=k)

    # Step 2: Get the FIRST chunk from the best article (contains the definition)
    best_doc_id = chunks[0]['doc_id']
    def_idx = doc_first_chunk[best_doc_id]
    definition_chunk = all_chunks[def_idx]

    # Step 3: Build context — definition chunk is primary, add second article if different
    context_parts = [definition_chunk]
    for c in chunks[1:]:
        if c['doc_id'] != best_doc_id:
            # Add a supporting chunk from a different article for extra context
            other_def_idx = doc_first_chunk[c['doc_id']]
            context_parts.append(all_chunks[other_def_idx])
            break

    context = "\n\n".join(context_parts)

    # Step 4: Generate answer
    prompt = f"""Using only the reference text below, write a clear 1-2 sentence answer.

Reference text:
{context}

Question: {query}
Answer:"""

    response = generate_text(prompt)

    return {
        'answer': response,
        'context': chunks,
        'definition_chunk': definition_chunk
    }

# Test RAG
query = "What is the difference between stocks and bonds?"

print("\U0001f4ac RAG RESPONSE")
print("=" * 60)
print(f"Query: \"{query}\"\n")

result = rag_answer(query)

print("Definition chunk used:")
print(wrap(result['definition_chunk'][:200]))

print(f"\nGenerated Answer:")
print(wrap(result['answer']))

💬 RAG RESPONSE
Query: "What is the difference between stocks and bonds?"

Definition chunk used:
   [Bond Market Definition] The bond market—often called the debt market, fixed-
   income market, or credit market—is the collective name given to all trades
   and issues of debt securities. Governments typic

Generated Answer:
   A stock (also known as equity) is a security that represents the ownership of
   a fraction


In [8]:
# Compare RAG vs No-RAG on multiple queries
comparison_queries = [
    "What is dollar-cost averaging?",
    "What is a mutual fund?",
    "What is an ETF?",
]

print("\U0001f4ca RAG vs NO-RAG COMPARISON")
print("=" * 60)

for query in comparison_queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 50)

    # Without RAG — model must rely entirely on its own training knowledge
    no_rag = generate_text(f"Answer in 1-2 sentences: {query}")
    print(f"  WITHOUT RAG:")
    print(wrap(no_rag))

    # With RAG — model gets Investopedia definitions as context
    rag_result = rag_answer(query)
    print(f"  WITH RAG:")
    print(wrap(rag_result['answer']))

print("\n\U0001f4a1 Notice how RAG answers include specific details from Investopedia!")
print("   Without RAG, the model gives vague or generic answers.")

📊 RAG vs NO-RAG COMPARISON

Query: "What is dollar-cost averaging?"
--------------------------------------------------
  WITHOUT RAG:
   dollar-cost averaging is the practice of saving for a set amount of money
   each and every month.
  WITH RAG:
   an investment strategy in which an investor divides up the total amount to be
   invested across periodic purchases of a target asset in an effort to reduce
   the impact of volatility

Query: "What is a mutual fund?"
--------------------------------------------------
  WITHOUT RAG:
   mutual fund is a type of investment vehicle that invests a portion of a
   company's assets in a variety of securities.
  WITH RAG:
   A type of financial vehicle made up of a pool of money collected from many
   investors to invest in securities like stocks, bonds, money market
   instruments, and other assets

Query: "What is an ETF?"
--------------------------------------------------
  WITHOUT RAG:
   An Exchange-Traded Fund (ETF) is a type of exchange-tr

In [9]:
from IPython.display import HTML, display

# Run queries and collect RAG vs No-RAG answers
demo_queries = [
    "What is dollar-cost averaging?",
    "What is a hedge fund?",
    "What is the difference between a bull and bear market?",
]

rows = ""
for q in demo_queries:
    rag_result = rag_answer(q)
    no_rag_result = generate_text(f"Answer in 1-2 sentences: {q}")

    rows += f"""
    <tr>
      <td style="padding:12px 16px; border-bottom:1px solid #333; color:#ccc; font-weight:600;
                 font-size:14px; vertical-align:top; width:18%;">{q}</td>
      <td style="padding:12px 16px; border-bottom:1px solid #333; background:#0d2818;
                 color:#4ecd94; font-size:13px; vertical-align:top; line-height:1.6;
                 width:41%; word-wrap:break-word; white-space:normal;">
        {rag_result['answer']}</td>
      <td style="padding:12px 16px; border-bottom:1px solid #333; background:#2a1215;
                 color:#ff6b6b; font-size:13px; vertical-align:top; line-height:1.6;
                 width:41%; word-wrap:break-word; white-space:normal;">
        {no_rag_result}</td>
    </tr>"""

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">
  <h2 style="color:white; text-align:center; margin-bottom:4px;">
    RAG vs No-RAG: Side-by-Side Comparison</h2>
  <p style="color:#999; text-align:center; margin-top:0; margin-bottom:16px; font-size:13px;">
    Same model (flan-t5-large), same questions &mdash; the only difference is retrieved context</p>
  <table style="width:100%; border-collapse:collapse; background:#1a1a2e;
                border-radius:12px; overflow:hidden; table-layout:fixed;">
    <thead>
      <tr>
        <th style="padding:14px 16px; background:#25254a; color:#aaa; text-align:left;
                   font-size:13px; width:18%;">Question</th>
        <th style="padding:14px 16px; background:#0a3d1f; color:#4ecd94; text-align:left;
                   font-size:14px; width:41%;">
          &#x2705; WITH RAG <span style="font-weight:normal; color:#4ecd9488;">
          &mdash; grounded in Investopedia</span></th>
        <th style="padding:14px 16px; background:#3d0a0f; color:#ff6b6b; text-align:left;
                   font-size:14px; width:41%;">
          &#x274c; WITHOUT RAG <span style="font-weight:normal; color:#ff6b6b88;">
          &mdash; model&rsquo;s training knowledge only</span></th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
  <p style="color:#666; text-align:center; margin-top:12px; font-size:12px;">
    flan-t5-large (780M params) &bull; BAAI/bge-small-en-v1.5 embeddings &bull;
    Investopedia financial terms ({len(df):,} articles)</p>
</div>
"""

display(HTML(html))

Question,✅ WITH RAG — grounded in Investopedia,❌ WITHOUT RAG — model’s training knowledge only
What is dollar-cost averaging?,an investment strategy in which an investor divides up the total amount to be invested across periodic purchases of a target asset in an effort to reduce the impact of volatility,dollar-cost averaging is the practice of saving for a set amount of money each and every month.
What is a hedge fund?,Hedge funds are alternative investments using pooled funds that employ different strategies to earn active returns,A hedge fund is a type of mutual fund.
What is the difference between a bull and bear market?,"Because prices of securities rise and fall essentially continuously during trading, the term ""bull market"" is typically reserved for extended periods in which a large portion of security prices are rising",Bull market Bull market may also refer to:


In [10]:
# Test multiple queries and evaluate quality
test_queries = [
    "What is a 401(k) plan?",
    "How does compound interest work?",
    "What is a hedge fund?",
    "What is the difference between a bull and bear market?",
    "What is an index fund?",
]

print("\U0001f4ca RAG EVALUATION")
print("=" * 60)

for query in test_queries:
    print(f"\nQ: {query}")
    result = rag_answer(query)
    print(f"A:")
    print(wrap(result['answer']))

    # Show retrieval quality
    top_score = 1 / (1 + result['context'][0]['distance'])
    print(f"   [Top retrieval score: {top_score:.2f}]")
    print("-" * 40)

📊 RAG EVALUATION

Q: What is a 401(k) plan?
A:
   tax-advantaged, defined-contribution retirement account
   [Top retrieval score: 0.83]
----------------------------------------

Q: How does compound interest work?
A:
   Compound interest (or compounding interest) is the interest on a loan or
   deposit calculated based on both the initial principal and the accumulated
   interest from previous periods. Thought to have originated in 17th-century
   Italy, compound interest can be thought of as "interest on interest," and
   will make a sum grow at a faster rate
   [Top retrieval score: 0.78]
----------------------------------------

Q: What is a hedge fund?
A:
   Hedge funds are alternative investments using pooled funds that employ
   different strategies to earn active returns
   [Top retrieval score: 0.79]
----------------------------------------

Q: What is the difference between a bull and bear market?
A:
   Because prices of securities rise and fall essentially continuously durin

In [11]:
from IPython.display import HTML, display

# Run queries and collect RAG vs No-RAG answers
demo_queries = [
    "What is dollar-cost averaging?",
    "What is a hedge fund?",
    "What is the difference between a bull and bear market?",
]

rows = ""
for q in demo_queries:
    rag_result = rag_answer(q)
    no_rag_result = generate_text(f"Answer in 1-2 sentences: {q}")

    rows += f"""
    <tr>
      <td style="padding:12px 16px; border-bottom:1px solid #333; color:#ccc; font-weight:600;
                 font-size:14px; vertical-align:top; width:18%;">{q}</td>
      <td style="padding:12px 16px; border-bottom:1px solid #333; background:#0d2818;
                 color:#4ecd94; font-size:13px; vertical-align:top; line-height:1.6;
                 width:41%; word-wrap:break-word; white-space:normal;">
        {rag_result['answer']}</td>
      <td style="padding:12px 16px; border-bottom:1px solid #333; background:#2a1215;
                 color:#ff6b6b; font-size:13px; vertical-align:top; line-height:1.6;
                 width:41%; word-wrap:break-word; white-space:normal;">
        {no_rag_result}</td>
    </tr>"""

html = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;">
  <h2 style="color:white; text-align:center; margin-bottom:4px;">
    RAG vs No-RAG: Side-by-Side Comparison</h2>
  <p style="color:#999; text-align:center; margin-top:0; margin-bottom:16px; font-size:13px;">
    Same model (flan-t5-large), same questions &mdash; the only difference is retrieved context</p>
  <table style="width:100%; border-collapse:collapse; background:#1a1a2e;
                border-radius:12px; overflow:hidden; table-layout:fixed;">
    <thead>
      <tr>
        <th style="padding:14px 16px; background:#25254a; color:#aaa; text-align:left;
                   font-size:13px; width:18%;">Question</th>
        <th style="padding:14px 16px; background:#0a3d1f; color:#4ecd94; text-align:left;
                   font-size:14px; width:41%;">
          &#x2705; WITH RAG <span style="font-weight:normal; color:#4ecd9488;">
          &mdash; grounded in Investopedia</span></th>
        <th style="padding:14px 16px; background:#3d0a0f; color:#ff6b6b; text-align:left;
                   font-size:14px; width:41%;">
          &#x274c; WITHOUT RAG <span style="font-weight:normal; color:#ff6b6b88;">
          &mdash; model&rsquo;s training knowledge only</span></th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
  <p style="color:#666; text-align:center; margin-top:12px; font-size:12px;">
    flan-t5-large (780M params) &bull; BAAI/bge-small-en-v1.5 embeddings &bull;
    Investopedia financial terms (1,000 articles)</p>
</div>
"""

display(HTML(html))

Question,✅ WITH RAG — grounded in Investopedia,❌ WITHOUT RAG — model’s training knowledge only
What is dollar-cost averaging?,an investment strategy in which an investor divides up the total amount to be invested across periodic purchases of a target asset in an effort to reduce the impact of volatility,dollar-cost averaging is the practice of saving for a set amount of money each and every month.
What is a hedge fund?,Hedge funds are alternative investments using pooled funds that employ different strategies to earn active returns,A hedge fund is a type of mutual fund.
What is the difference between a bull and bear market?,"Because prices of securities rise and fall essentially continuously during trading, the term ""bull market"" is typically reserved for extended periods in which a large portion of security prices are rising",Bull market Bull market may also refer to:
